# Comparaison des modèles et décisions de modélisation

Ce notebook justifie le modèle déployé par `src/train.py` : du plus simple au plus complexe, chaque candidat est
évalué de la même façon, sur la métrique qui compte pour l'atelier.

## Une métrique événementielle

Le système évalue chaque machine toutes les 30 minutes. Compter les fausses alertes par pas pénaliserait plusieurs
fois le même dérangement : une machine en alerte 10 h d'affilée, c'est une seule visite du technicien. On suit donc :

- le **recall événementiel** : part des pannes précédées d'au moins une alerte dans les 120 h ;
- les **fausses alertes par machine et par jour**, comptées en épisodes (une suite d'alertes = une intervention ;
  au-delà de 24 h, l'épisode est redécoupé pour qu'un modèle bloqué en alerte ne passe pas pour parfait).

Le seuil de chaque modèle est celui qui maximise le recall sous un budget de **0,2 fausse alerte par machine et par
jour** (un dérangement inutile tous les 5 jours par machine).

## Protocole

- Trois fenêtres de validation (T2, T3 et T4 2024), chacune entraînée sur ce qui précède, avec un tampon de 120 h
  pour que la cible ne déborde pas sur la période suivante.
- Test sur 2025, avec le seuil choisi sur la dernière fenêtre, appliqué tel quel.
- Une machine sur cinq n'est jamais vue à l'entraînement : elle mesure la généralisation à une machine neuve.

In [1]:
import sys
import time

sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from src.config import CIBLE
from src.donnees import FEATURES, PLIS, charger, decouper, decouper_pli, preparer_X
from src.evaluation import (
    BUDGET_FAUSSES_ALERTES_PAR_MACHINE_JOUR,
    PAS_HORIZON,
    bilan_economique,
    grille_seuils,
    metriques_evenementielles,
    trouver_seuil_pour_budget_fa,
)
from src.train import nouveau_modele

df = charger()
_, _, test = decouper(df)

## Les candidats, du plus simple au plus complexe

In [2]:
numeriques = [f for f in FEATURES if f != "type_machine"]
encodeur = lambda numerique: make_column_transformer(
    (numerique, numeriques), (OneHotEncoder(handle_unknown="ignore"), ["type_machine"])
)
CANDIDATS = {
    "Régression logistique": lambda: make_pipeline(
        encodeur(make_pipeline(SimpleImputer(strategy="median"), StandardScaler())),
        LogisticRegression(max_iter=3000, class_weight="balanced"),
    ),
    "Arbre de décision": lambda: make_pipeline(
        encodeur("passthrough"), DecisionTreeClassifier(max_depth=6, min_samples_leaf=500, random_state=42)
    ),
    "HistGradientBoosting": lambda: HistGradientBoostingClassifier(
        max_iter=150, learning_rate=0.1, max_leaf_nodes=15, min_samples_leaf=80, categorical_features="from_dtype", random_state=42
    ),
    "LightGBM (déployé)": nouveau_modele,
}

In [3]:
def recall_au_budget(fenetre, scores):
    return trouver_seuil_pour_budget_fa(
        fenetre, scores, grille_seuils(scores), PAS_HORIZON, BUDGET_FAUSSES_ALERTES_PAR_MACHINE_JOUR
    )


def evaluer(fabrique):
    ligne, pannes_validation = {}, 0
    for i, (debut, fin) in enumerate(PLIS, start=1):
        train, validation = decouper_pli(df, debut, fin)
        depart = time.perf_counter()
        train = train[train["etat"] == "marche"]  # le modèle n'apprend que sur les machines en marche
        modele = fabrique().fit(preparer_X(train), train[CIBLE])
        ligne["entrainement_s"] = time.perf_counter() - depart
        seuil, m = recall_au_budget(validation, modele.predict_proba(preparer_X(validation))[:, 1])
        ligne[f"recall_T{i + 1}_2024"] = m["recall_evenementiel"]
        pannes_validation += m["nb_pannes"]
    # Protocole de déploiement : le seuil de la dernière fenêtre, appliqué tel quel au test 2025
    # (l'évaluation n'alerte jamais une machine arrêtée)
    m = metriques_evenementielles(test, modele.predict_proba(preparer_X(test))[:, 1], seuil, PAS_HORIZON)
    recall_validation = np.mean([ligne[f"recall_T{i + 1}_2024"] for i in range(1, len(PLIS) + 1)])
    return ligne | {
        "recall_validation": recall_validation,
        # Écart-type binomial : sous ~2 écarts-types, deux modèles ne sont pas départagés
        "ecart_type_validation": np.sqrt(recall_validation * (1 - recall_validation) / pannes_validation),
        "recall_test_2025": m["recall_evenementiel"],
        "ecart_type_test": np.sqrt(m["recall_evenementiel"] * (1 - m["recall_evenementiel"]) / m["nb_pannes"]),
        "recall_preavis_4h": m["recall_preavis_4h"],
        "precision_alertes": m["precision_alertes"],
        "fa_test": m["taux_fausses_alertes_par_machine_jour"],
        "gain_net_M_fcfa": bilan_economique(test, m)["gain_net"] / 1e6,
    }


resultats = pd.DataFrame({nom: evaluer(fabrique) for nom, fabrique in CANDIDATS.items()}).T
colonnes = ["recall_validation", "ecart_type_validation", "recall_test_2025", "ecart_type_test", "recall_preavis_4h"]
colonnes += ["precision_alertes", "fa_test", "gain_net_M_fcfa", "entrainement_s"]
resultats[colonnes].astype(float).round(3)

,recall_validation,ecart_type_validation,recall_test_2025,ecart_type_test,recall_preavis_4h,precision_alertes,fa_test,gain_net_M_fcfa,entrainement_s
Régression logistique,0.723,0.055,0.470,0.047,0.409,0.020,0.144,-3.502,4.405
Arbre de décision,0.691,0.057,0.530,0.047,0.443,0.014,0.239,-2.486,7.003
HistGradientBoosting,0.769,0.052,0.626,0.045,0.539,0.017,0.223,0.015,4.170
LightGBM (déployé),0.735,0.055,0.643,0.045,0.548,0.019,0.210,-0.634,1.933


## Lecture

Le boosting annonce le plus de pannes : LightGBM 64 % sur 2025, dont 55 % avec au moins 4 h de préavis. Entre les
deux boostings, l'écart reste dans l'incertitude (HistGradientBoosting devance en validation, LightGBM au test) :
LightGBM est conservé, plus rapide et plus léger.

En FCFA, le bilan reste fragile : environ une alerte sur cinquante précède une panne, quel que soit le modèle, et le
gain net de LightGBM sur 2025 est proche de zéro. Les pistes : un seuil plus haut (moins d'alertes, plus sûres),
choisi sur une validation plus longue que les 22 pannes de fin 2024, ou une inspection plus courte qu'une heure.

## Ce qui compte pour le modèle

In [4]:
train, _, _ = decouper(df)
lgbm = nouveau_modele().fit(preparer_X(train), train[CIBLE])
pd.Series(lgbm.feature_importances_, index=FEATURES).sort_values(ascending=False).head(10)

heures_depuis_changement_outil         535
heures_depuis_maintenance              485
vibration_max_24h                      200
concentration_poussiere_max_24h        143
ecart_thermique_max_24h                114
temperature_process_max_24h            111
type_machine                            91
couple_max_24h                          91
concentration_poussiere_moyenne_24h     71
vibration_moyenne_24h                   67
dtype: int32

## Décisions de modélisation

- **Seulement les machines en marche** : à l'arrêt (week-ends), en panne ou en délestage, les capteurs tombent à
  zéro. Le modèle n'apprend que sur les relevés en marche et n'alerte jamais une machine qui ne tourne pas ; la cible,
  elle, est calculée sur toute la chronologie. Gain : quatre points de recall, et plus d'alertes sur des machines arrêtées.
- **Budget de fausses alertes justifié par les coûts** (`src/usine.py`) : une panne annoncée devient un arrêt planifié
  de 2 h sans rebut (environ 235 000 FCFA évités en moyenne), une alerte coûte une heure de technicien (3 250 FCFA).
  Sur 5 jours d'horizon, une panne possible peut coûter jusqu'à 5 inspections : le seuil rentable vaut donc
  5 × 3 250 / 235 000 ≈ 6,9 %, presque exactement celui du budget de 0,2 fausse alerte par machine et par jour.
  Le gain net reste proche de zéro : le modèle n'est rentable que si une inspection prend moins d'une heure.
- **Promotion décidée sur la validation** : le @champion n'est remplacé que si le gain net en validation ne baisse
  pas ; le test 2025 ne sert qu'au rapport.
- **Pas de standardisation, catégorie native** : un modèle à base d'arbres est insensible aux transformations
  monotones ; LightGBM gère les valeurs manquantes et la gamme (`type_machine`) nativement. Les catégories sont
  fixées (L, M, H) pour qu'une machine prédite seule soit lue comme à l'entraînement.
- **Calibration sigmoïde sur les log-odds** : elle rend les probabilités interprétables (sur 100 machines à 10 %,
  une dizaine tombe en panne dans les 5 jours). Ajustée sur la probabilité brute, bornée à 0, elle imposait un
  plancher d'environ 2,8 % à toutes les machines saines ; sur les log-odds, ce plancher disparaît, au même recall.
- **OSF (surcharge mécanique) reste le point faible**, faute d'exemples. Trois pistes testées puis écartées : un
  classifieur dédié (moins bon que le modèle commun), des horizons plus courts pour HDF et OSF (moins d'exemples
  positifs), et une surpondération des pannes OSF (neutre au mieux).
- **50 machines** : un essai à 100 machines déplaçait les résultats par mode, signe d'un effet d'échantillon et non d'architecture.